# Agent Evaluation

This notebook adds occupied thermal comfort and cross-building fairness to the carbon-aware MARLISA reward, then compares all five approaches developed so far:

1. `BasicRBC`;
2. independent SAC;
3. MARLISA;
4. carbon-aware MARLISA; and
5. fair carbon-aware MARLISA.

The main question is whether the fair controller can retain useful carbon reduction while substantially improving the experience of the worst-served building.


In [ ]:
from __future__ import annotations
import sys
import json
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots


PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUTPUTS_ROOT = PROJECT_ROOT / 'outputs'


CONTROLLERS = {
    "rbc": {"label": "RBC", "color": "#64748B"},
    "independent_sac": {"label": "Independent SAC", "color": "#2563EB"},
    "marlisa": {"label": "MARLISA", "color": "#0F9D8A"},
    "carbon_aware_marlisa": { "label": "Carbon-aware MARLISA", "color": "#F59E0B"},
    "fair_carbon_aware_marlisa": {"label": "Fair carbon-aware MARLISA","color": "#7C3AED"},
}
model_list = ["rbc", "independent_sac", "marlisa", "carbon_aware_marlisa", "fair_carbon_aware_marlisa"]
COMFORT_BAND_CELSIUS = 2.0

# Run all controller experiments

Use the configuration cell below when you want to regenerate every canonical output folder with one shared training budget. `EPISODES` applies to independent SAC and the three MARLISA variants. RBC has no learning episodes, so it is rerun once over the same simulation period and seed.

The final episode of each learning controller is deterministic evaluation. Therefore, `EPISODES = 10` means nine non-deterministic training/exploration episodes followed by one deterministic evaluation episode. Running this section overwrites the current canonical files under `outputs/`.


In [65]:
# Experiment configuration: change these values before running this section.
RUN_ALL_MODELS = True  # Change to True to retrain and overwrite canonical outputs.
EPISODES = 10         # Must be at least 3 for the learning controllers.
RANDOM_SEED = 42

# Selections used by the evaluation charts later in the notebook.
FOCUS_MODEL = "fair_carbon_aware_marlisa"
FOCUS_BUILDING = "Building_1"
SIMULATION_DAY = 1
SAVE_CHART_HTML = True

SCHEMA = PROJECT_ROOT / "notebooks" / "data" / "schema.json"

if EPISODES < 3:
    raise ValueError("EPISODES must be at least 3: exploration, training, and deterministic evaluation.")
if not SCHEMA.is_file():
    raise FileNotFoundError(f"CityLearn schema was not found: {SCHEMA}")

print(f"Learning-controller episodes: {EPISODES}")
print(f"Random seed: {RANDOM_SEED}")
print(f"Canonical output root: {OUTPUTS_ROOT}")
print(f"Run enabled: {RUN_ALL_MODELS}")


Learning-controller episodes: 10
Random seed: 42
Canonical output root: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs
Run enabled: True


In [66]:
import gc
from time import perf_counter



def run_all_controller_experiments(
    episodes: int,
    random_seed: int,
) -> pd.DataFrame:
    """Run all five controllers sequentially and overwrite canonical outputs."""
    if episodes < 3:
        raise ValueError("episodes must be at least 3")

    # Delay CityLearn/Torch imports until training is explicitly enabled.
    from src.run_rbc import run_basic_rbc
    from src.run_independent_sac import run_independent_sac
    from src.run_marlisa import run_marlisa
    from src.run_carbon_aware_marlisa import run_carbon_aware_marlisa
    from src.run_fair_carbon_aware_marlisa import (
        run_fair_carbon_aware_marlisa,
    )

    jobs = [
        (
            "rbc",
            lambda: run_basic_rbc(
                schema=SCHEMA,
                output_directory=OUTPUTS_ROOT / "rbc",
                random_seed=random_seed,
            ),
        ),
        (
            "independent_sac",
            lambda: run_independent_sac(
                schema=SCHEMA,
                output_directory=OUTPUTS_ROOT / "independent_sac",
                episodes=episodes,
                random_seed=random_seed,
            ),
        ),
        (
            "marlisa",
            lambda: run_marlisa(
                schema=SCHEMA,
                output_directory=OUTPUTS_ROOT / "marlisa",
                episodes=episodes,
                random_seed=random_seed,
                reward="marl",
            ),
        ),
        (
            "carbon_aware_marlisa",
            lambda: run_carbon_aware_marlisa(
                schema=SCHEMA,
                output_directory=OUTPUTS_ROOT / "carbon_aware_marlisa",
                episodes=episodes,
                random_seed=random_seed,
            ),
        ),
        (
            "fair_carbon_aware_marlisa",
            lambda: run_fair_carbon_aware_marlisa(
                schema=SCHEMA,
                output_directory=OUTPUTS_ROOT / "fair_carbon_aware_marlisa",
                episodes=episodes,
                random_seed=random_seed,
                carbon_weight=1.0,
                comfort_weight=0.25,
                fairness_weight=0.25,
                comfort_band=1.0,
            ),
        ),
    ]

    records = []
    overall_start = perf_counter()
    for model, run_job in jobs:
        print(f"\n{'=' * 72}\nRunning {CONTROLLERS[model]['label']}\n{'=' * 72}")
        start = perf_counter()
        result = run_job()
        elapsed_minutes = (perf_counter() - start) / 60.0
        records.append(
            {
                "model": model,
                "episodes": 1 if model == "rbc" else episodes,
                "seed": random_seed,
                "minutes": elapsed_minutes,
                "output_directory": str(result["output_directory"]),
                "status": "completed",
            }
        )
        print(f"Completed {model} in {elapsed_minutes:.1f} minutes")

        # Avoid retaining every trained agent and environment in notebook memory.
        del result
        gc.collect()

    total_minutes = (perf_counter() - overall_start) / 60.0
    print(f"\nAll controllers completed in {total_minutes:.1f} minutes.")
    return pd.DataFrame(records)


if RUN_ALL_MODELS:
    experiment_run_log = run_all_controller_experiments(
        episodes=EPISODES,
        random_seed=RANDOM_SEED,
    )
    display(experiment_run_log)
    print("The cells below will now reload the updated output files.")
else:
    print(
        "Training was skipped. Set RUN_ALL_MODELS = True and run this cell "
        "to regenerate all canonical outputs."
    )



Running RBC
Completed rbc in 0.1 minutes

Running Independent SAC
Completed independent_sac in 8.3 minutes

Running MARLISA
Completed marlisa in 11.1 minutes

Running Carbon-aware MARLISA
Completed carbon_aware_marlisa in 10.8 minutes

Running Fair carbon-aware MARLISA
Completed fair_carbon_aware_marlisa in 9.9 minutes

All controllers completed in 40.4 minutes.


,model,episodes,seed,minutes,output_directory,status
0,rbc,1,42,0.142121,C:\Users\tanka\Dropbox\08-1 NUS Master in Comp...,completed
1,independent_sac,10,42,8.338426,C:\Users\tanka\Dropbox\08-1 NUS Master in Comp...,completed
2,marlisa,10,42,11.109040,C:\Users\tanka\Dropbox\08-1 NUS Master in Comp...,completed
3,carbon_aware_marlisa,10,42,10.800062,C:\Users\tanka\Dropbox\08-1 NUS Master in Comp...,completed
4,fair_carbon_aware_marlisa,10,42,9.949766,C:\Users\tanka\Dropbox\08-1 NUS Master in Comp...,completed


The cells below will now reload the updated output files.


# Load Data

In [77]:
@dataclass
class RunData:
    """Files loaded for one saved controller run."""

    key: str
    label: str
    color: str
    kpis: pd.DataFrame
    observations: pd.DataFrame
    actions: pd.DataFrame
    metadata: dict[str, Any]

def load_runs(outputs_root: Path = OUTPUTS_ROOT) -> dict[str, RunData]:
    """Load canonical runs, skipping controllers without KPI files."""
    runs: dict[str, RunData] = {}

    for key, display in CONTROLLERS.items():
        folder = outputs_root / key
        kpi_path = folder / "kpis.csv"
        if not kpi_path.is_file():
            continue

        metadata_path = folder / "metadata.json"
        metadata = (
            json.loads(metadata_path.read_text(encoding="utf-8"))
            if metadata_path.is_file()
            else {}
        )
        runs[key] = RunData(
            key=key,
            label=display["label"],
            color=display["color"],
            kpis=pd.read_csv(kpi_path),
            observations= pd.read_csv(folder / "observations.csv"),
            actions= pd.read_csv(folder / "actions.csv"),
            metadata=metadata,
        )

    if not runs:
        raise FileNotFoundError(
            f"No canonical controller KPI files were found under {outputs_root}."
        )

    return runs
##-------------------------##

runs = load_runs()


# Helper Functions

In [78]:
def build_summary(runs: dict[str, RunData]) -> pd.DataFrame:
    """Calculate one carbon and comfort summary row for every run."""
    rows: list[dict[str, Any]] = []

    for key, run in runs.items():
        district_carbon = run.kpis.loc[
            (run.kpis["cost_function"] == "carbon_emissions_total")
            & (run.kpis["level"] == "district"),
            "value",
        ]
        discomfort = run.kpis.loc[
            (run.kpis["cost_function"] == "discomfort_proportion")
            & (run.kpis["level"] == "building"),
            ["name", "value"],
        ].dropna()
        if len(district_carbon) != 1 or discomfort.empty:
            continue

        worst_value = float(discomfort["value"].max())
        best_value = float(discomfort["value"].min())
        worst_names = discomfort.loc[
            np.isclose(
                discomfort["value"], worst_value, rtol=0.0, atol=1e-12
            ),
            "name",
        ].astype(str)
        rows.append(
            {
                "model": key,
                "controller": run.label,
                "color": run.color,
                "carbon_ratio": float(district_carbon.iloc[0]),
                "average_discomfort": float(discomfort["value"].mean()),
                "best_discomfort": best_value,
                "worst_discomfort": worst_value,
                "discomfort_gap": worst_value - best_value,
                "worst_building": ", ".join(worst_names),
                "seed": run.metadata.get("random_seed"),
                "episodes": run.metadata.get("episodes"),
            }
        )

    return pd.DataFrame(rows)
##-------------------------##

def building_discomfort(runs: dict[str, RunData]) -> pd.DataFrame:
    """Return building-level discomfort rows for all runs."""
    frames: list[pd.DataFrame] = []
    for key, run in runs.items():
        values = run.kpis.loc[
            (run.kpis["cost_function"] == "discomfort_proportion")
            & (run.kpis["level"] == "building"),
            ["name", "value"],
        ].copy()
        values["model"] = key
        values["controller"] = run.label
        frames.append(values)
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()

##-------------------------##

summary = build_summary(runs)

In [79]:
def _base_layout(fig: go.Figure, *, height: int = 390) -> go.Figure:
    fig.update_layout(
        height=height,
        margin=dict(l=54, r=24, t=58, b=50),
        paper_bgcolor="#FFFFFF",
        plot_bgcolor="#F8FAFC",
        font=dict(family="Inter, Arial, sans-serif", color="#172033"),
        hoverlabel=dict(bgcolor="#172033", font_color="#FFFFFF"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
    )
    fig.update_xaxes(gridcolor="#E2E8F0", zeroline=False)
    fig.update_yaxes(gridcolor="#E2E8F0", zeroline=False)
    return fig


## a) Summary Statistics


In [80]:
summary = build_summary(runs)
req=['model',  'carbon_ratio', 'average_discomfort',
       'best_discomfort', 'worst_discomfort', 'discomfort_gap',
       'worst_building', 'episodes']
display(summary[req])

,model,carbon_ratio,average_discomfort,best_discomfort,worst_discomfort,discomfort_gap,worst_building,episodes
0,rbc,1.889587,0.946599,0.908411,0.974719,0.066308,Building_1,NaN
1,independent_sac,0.413081,0.979771,0.977570,0.981742,0.004171,Building_1,10.0
2,marlisa,0.619078,0.617769,0.463551,0.903090,0.439538,Building_1,10.0
3,carbon_aware_marlisa,0.419089,0.979771,0.977570,0.981742,0.004171,Building_1,10.0
4,fair_carbon_aware_marlisa,0.845985,0.136132,0.097196,0.207865,0.110669,Building_1,10.0


## b) Carbon Performance against the reference

In [81]:
def carbon_bar_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Carbon ratio bars with the no-control reference marked at 1.0."""
    frame = summary.sort_values("carbon_ratio", ascending=True)
    colors = [
        CONTROLLERS[m]["color"] if m == focus else "#CBD5E1"
        for m in frame["model"]
    ]
    fig = go.Figure(
        go.Bar(
            x=frame["carbon_ratio"],
            y=frame["controller"],
            orientation="h",
            marker_color=colors,
            text=frame["carbon_ratio"].map(lambda x: f"{x:.3f}"),
            textposition="outside",
            customdata=(1.0 - frame["carbon_ratio"]) * 100.0,
            hovertemplate=(
                "%{y}<br>Carbon ratio: %{x:.3f}<br>"
                "Change vs no-control: %{customdata:+.1f}%<extra></extra>"
            ),
        )
    )
    fig.add_vline(
        x=1.0,
        line_dash="dash",
        line_color="#DC2626",
        annotation_text="No-control reference",
        annotation_position="top right",
    )
    fig.update_layout(title="Carbon performance against the reference", showlegend=False)
    fig.update_xaxes(title="Carbon-emissions ratio · lower is better", rangemode="tozero")
    fig.update_yaxes(title=None)
    return _base_layout(fig)

carbon_bar_figure_output = carbon_bar_figure(summary, focus="fair_carbon_aware_marlisa")
carbon_bar_figure_output

## c) The Central Trade off: Carbon Emissions vs Discomfort

In [82]:
def tradeoff_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Carbon versus worst-building discomfort trade-off."""
    fig = go.Figure()
    for row in summary.itertuples(index=False):
        selected = row.model == focus
        fig.add_trace(
            go.Scatter(
                x=[row.carbon_ratio],
                y=[row.worst_discomfort],
                mode="markers+text",
                text=[row.controller],
                textposition="top center",
                marker=dict(
                    size=18 if selected else 13,
                    color=row.color,
                    line=dict(color="#FFFFFF", width=3 if selected else 1.5),
                    opacity=1.0 if selected else 0.78,
                ),
                customdata=[[row.average_discomfort, row.discomfort_gap]],
                hovertemplate=(
                    "%{text}<br>Carbon ratio: %{x:.3f}<br>"
                    "Worst discomfort: %{y:.1%}<br>"
                    "Average discomfort: %{customdata[0]:.1%}<br>"
                    "Gap: %{customdata[1]:.1%}<extra></extra>"
                ),
                showlegend=False,
            )
        )
    fig.add_hline(y=0.5, line_dash="dash", line_color="#DC2626")
    fig.add_vline(x=1.0, line_dash="dash", line_color="#DC2626")
    fig.add_annotation(
        x=0.02,
        y=0.04,
        xref="paper",
        yref="paper",
        text="Preferred Quadrant (direction ↙)",
        showarrow=False,
        font=dict(color="#166534", size=12),
        bgcolor="#DCFCE7",
        borderpad=5,
    )
    fig.update_layout(title="The central trade-off: carbon versus protected comfort")
    fig.update_xaxes(title="Carbon-emissions ratio", rangemode="tozero")
    fig.update_yaxes(
        title="Worst-building discomfort",
        range=[0, 1.2],
        tickformat=".0%",
    )
    return _base_layout(fig)

tradeoff_figure_output = tradeoff_figure(summary, focus="fair_carbon_aware_marlisa")
tradeoff_figure_output

## d) Which Building experience discomfort?

In [83]:
def discomfort_by_building_figure(
    runs: dict[str, RunData], focus: str
) -> go.Figure:
    """Grouped building-level discomfort chart."""
    values = building_discomfort(runs)
    fig = go.Figure()
    for key, run in runs.items():
        part = values.loc[values["model"] == key]
        fig.add_trace(
            go.Bar(
                x=part["name"],
                y=part["value"],
                name=run.label,
                marker_color=run.color,
                opacity=1.0 if key == focus else 0.55,
                hovertemplate="%{x}<br>Discomfort: %{y:.1%}<extra>"
                + run.label
                + "</extra>",
            )
        )
    fig.update_layout(
        title="Who experiences discomfort?",
        barmode="group",
    )
    fig.update_xaxes(title=None)
    fig.update_yaxes(
        title="Occupied time outside comfort band",
        range=[0, 1.03],
        tickformat=".0%",
    )
    return _base_layout(fig, height=420)

discomfort_by_building_figure_output = discomfort_by_building_figure(runs, focus="fair_carbon_aware_marlisa")
discomfort_by_building_figure_output

In [84]:
def comfort_range_figure(summary: pd.DataFrame, focus: str) -> go.Figure:
    """Best-to-worst comfort ranges with average markers."""
    frame = summary.sort_values("average_discomfort", ascending=False)
    fig = go.Figure()
    for row in frame.itertuples(index=False):
        opacity = 1.0 if row.model == focus else 0.48
        fig.add_trace(
            go.Scatter(
                x=[row.best_discomfort, row.worst_discomfort],
                y=[row.controller, row.controller],
                mode="lines+markers",
                line=dict(color=row.color, width=7),
                marker=dict(color=row.color, size=11),
                opacity=opacity,
                hovertemplate="%{y}<br>%{x:.1%}<extra></extra>",
                showlegend=False,
            )
        )
        fig.add_trace(
            go.Scatter(
                x=[row.average_discomfort],
                y=[row.controller],
                mode="markers",
                marker=dict(
                    color="#FFFFFF",
                    line=dict(color=row.color, width=3),
                    size=13,
                    symbol="diamond",
                ),
                opacity=opacity,
                hovertemplate=(
                    f"{row.controller}<br>Average: "
                    "%{x:.1%}<extra></extra>"
                ),
                showlegend=False,
            )
        )
    fig.add_annotation(
        x=0.99,
        y=1.12,
        xref="paper",
        yref="paper",
        text="Ends = best/worst · diamond = average",
        showarrow=False,
        font=dict(size=11, color="#64748B"),
    )
    fig.update_layout(title="Comfort level and inequality must be read together")
    fig.update_xaxes(
        title="Building discomfort proportion",
        range=[0, 1.03],
        tickformat=".0%",
    )
    fig.update_yaxes(title=None)
    return _base_layout(fig, height=420)

comfort_range_figure_output = comfort_range_figure(summary, focus="fair_carbon_aware_marlisa")
comfort_range_figure_output

## e) Time-series evidence and building replay

The aggregate KPIs establish the result. These charts explain when carbon emissions accumulate, when the selected building violates the occupied comfort band, and how one day's temperature trajectory relates to the recorded controller actions.


In [85]:
def _empty_figure(message: str) -> go.Figure:
    fig = go.Figure()
    fig.add_annotation(
        x=0.5, y=0.5, xref="paper", yref="paper",
        text=message, showarrow=False,
        font=dict(size=15, color="#64748B"),
    )
    fig.update_xaxes(visible=False)
    fig.update_yaxes(visible=False)
    return _base_layout(fig)


def cumulative_carbon_figure(
    runs: dict[str, RunData], focus: str
) -> go.Figure:
    """Reconstruct cumulative operational emissions from observations."""
    fig = go.Figure()
    for key, run in runs.items():
        if run.observations.empty:
            continue
        frame = run.observations.copy()
        frame["emissions"] = (
            frame["net_electricity_consumption"].clip(lower=0.0)
            * frame["carbon_intensity"]
        )
        series = frame.groupby("time_step", sort=True)["emissions"].sum().cumsum()
        fig.add_trace(
            go.Scatter(
                x=series.index,
                y=series.values,
                mode="lines",
                name=run.label,
                line=dict(color=run.color, width=4 if key == focus else 2),
                opacity=1.0 if key == focus else 0.58,
                hovertemplate=(
                    "Time step %{x}<br>Cumulative emissions: "
                    "%{y:.2f} kgCO2<extra>" + run.label + "</extra>"
                ),
            )
        )
    fig.update_layout(title="How emissions accumulate through the evaluation period")
    fig.update_xaxes(title="Simulation time step")
    fig.update_yaxes(title="Cumulative operational emissions (kgCO2)")
    return _base_layout(fig, height=430)


def comfort_heatmap_figure(run: RunData, building: str) -> go.Figure:
    """Show occupied temperature deviation outside the evaluation comfort band."""
    frame = run.observations.loc[
        run.observations["building"] == building
    ].copy()
    if frame.empty:
        return _empty_figure("No observation history is available for this building.")

    frame["simulation_day"] = frame["time_step"] // 24 + 1
    frame["display_hour"] = frame["time_step"] % 24 + 1
    frame["outside_band"] = (
        (
            frame["indoor_dry_bulb_temperature"]
            - frame["indoor_dry_bulb_temperature_cooling_set_point"]
        ).abs()
        - COMFORT_BAND_CELSIUS
    ).clip(lower=0.0)
    frame.loc[frame["occupant_count"] <= 0, "outside_band"] = np.nan
    grid = frame.pivot(
        index="simulation_day", columns="display_hour", values="outside_band"
    )

    fig = go.Figure(
        go.Heatmap(
            z=grid.values, x=grid.columns, y=grid.index,
            colorscale=[
                [0.0, "#ECFDF5"], [0.15, "#FDE68A"],
                [0.5, "#FB923C"], [1.0, "#B91C1C"],
            ],
            colorbar=dict(title="degrees outside<br>band"),
            hovertemplate=(
                "Day %{y}<br>Hour %{x}<br>Deviation: %{z:.2f} C<extra></extra>"
            ),
            hoverongaps=False,
        )
    )
    fig.update_layout(title=f"When occupied comfort violations occur - {building}")
    fig.update_xaxes(title="Hour of day", dtick=3)
    fig.update_yaxes(title="Simulation day", autorange="reversed")
    return _base_layout(fig, height=430)


def building_replay_figure(
    run: RunData, building: str, simulation_day: int
) -> go.Figure:
    """Plot one day of temperatures and normalized controller actions."""
    start = (int(simulation_day) - 1) * 24
    end = start + 23
    observations = run.observations.loc[
        (run.observations["building"] == building)
        & run.observations["time_step"].between(start, end)
    ].copy()
    actions = run.actions.loc[
        (run.actions["building"] == building)
        & run.actions["time_step"].between(start, end)
    ].copy()
    if observations.empty:
        return _empty_figure("No replay data are available for this selection.")

    fig = make_subplots(
        rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.12,
        subplot_titles=("Indoor temperature and comfort band", "Recorded actions"),
    )
    hours = observations["time_step"] % 24 + 1
    setpoint = observations["indoor_dry_bulb_temperature_cooling_set_point"]

    fig.add_trace(
        go.Scatter(
            x=hours, y=setpoint - COMFORT_BAND_CELSIUS,
            mode="lines", line=dict(width=0), hoverinfo="skip",
            showlegend=False,
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hours, y=setpoint + COMFORT_BAND_CELSIUS,
            mode="lines", line=dict(width=0), fill="tonexty",
            fillcolor="rgba(37, 99, 235, 0.12)",
            name="+/- 2 C comfort band", hoverinfo="skip",
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hours, y=setpoint, mode="lines", name="Cooling setpoint",
            line=dict(color="#64748B", width=2, dash="dash"),
        ),
        row=1, col=1,
    )
    fig.add_trace(
        go.Scatter(
            x=hours, y=observations["indoor_dry_bulb_temperature"],
            mode="lines+markers", name="Indoor temperature",
            line=dict(color=run.color, width=3), marker=dict(size=5),
            customdata=observations[["occupant_count"]],
            hovertemplate=(
                "Hour %{x}<br>Temperature: %{y:.2f} C<br>"
                "Occupants: %{customdata[0]:.0f}<extra></extra>"
            ),
        ),
        row=1, col=1,
    )

    action_colors = {
        "dhw_storage": "#F59E0B",
        "electrical_storage": "#0F9D8A",
        "cooling_device": "#2563EB",
    }
    action_labels = {
        "dhw_storage": "Hot-water storage",
        "electrical_storage": "Battery",
        "cooling_device": "Cooling device",
    }
    for action_name, part in actions.groupby("action", sort=False):
        fig.add_trace(
            go.Scatter(
                x=part["time_step"] % 24 + 1, y=part["value"],
                mode="lines", name=action_labels.get(action_name, action_name),
                line=dict(color=action_colors.get(action_name, "#334155"), width=2),
                hovertemplate=(
                    "Hour %{x}<br>Normalized action: %{y:.3f}<extra></extra>"
                ),
            ),
            row=2, col=1,
        )

    fig.update_layout(
        title=f"Building replay - {run.label} - {building} - day {simulation_day}",
        height=620, margin=dict(l=60, r=24, t=78, b=50),
        paper_bgcolor="#FFFFFF", plot_bgcolor="#F8FAFC",
        font=dict(family="Inter, Arial, sans-serif", color="#172033"),
        legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0),
        hovermode="x unified",
    )
    fig.update_xaxes(title="Hour of day", dtick=3, row=2, col=1)
    fig.update_yaxes(title="Temperature (C)", gridcolor="#E2E8F0", row=1, col=1)
    fig.update_yaxes(
        title="Normalized action", range=[-1.05, 1.05],
        gridcolor="#E2E8F0", row=2, col=1,
    )
    return fig


if FOCUS_MODEL not in runs:
    raise KeyError(f"No saved run was found for {FOCUS_MODEL!r}.")
if FOCUS_BUILDING not in runs[FOCUS_MODEL].metadata.get("buildings", []):
    raise KeyError(f"{FOCUS_BUILDING!r} is not available in {FOCUS_MODEL!r}.")

cumulative_carbon_output = cumulative_carbon_figure(runs, FOCUS_MODEL)
comfort_heatmap_output = comfort_heatmap_figure(
    runs[FOCUS_MODEL], FOCUS_BUILDING
)
building_replay_output = building_replay_figure(
    runs[FOCUS_MODEL], FOCUS_BUILDING, SIMULATION_DAY
)

display(cumulative_carbon_output)
display(comfort_heatmap_output)
display(building_replay_output)


## f) Save the consolidated evaluation outputs

The controller runners have already updated their own CSV, JSON, and checkpoint files. This cell additionally saves the consolidated KPI summary and, when enabled, standalone interactive HTML versions of all evaluation charts.


In [86]:
evaluation_figures = {
    "01_carbon_ratio": carbon_bar_figure(summary, FOCUS_MODEL),
    "02_carbon_comfort_tradeoff": tradeoff_figure(summary, FOCUS_MODEL),
    "03_building_discomfort": discomfort_by_building_figure(runs, FOCUS_MODEL),
    "04_comfort_range": comfort_range_figure(summary, FOCUS_MODEL),
    "05_cumulative_carbon": cumulative_carbon_output,
    "06_comfort_heatmap": comfort_heatmap_output,
    "07_building_replay": building_replay_output,
}

summary_path = OUTPUTS_ROOT / "evaluation_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"Saved summary: {summary_path}")

if SAVE_CHART_HTML:
    chart_directory = OUTPUTS_ROOT / "evaluation_charts"
    chart_directory.mkdir(parents=True, exist_ok=True)
    for chart_name, figure in evaluation_figures.items():
        chart_path = chart_directory / f"{chart_name}.html"
        figure.write_html(
            chart_path,
            include_plotlyjs="cdn",
            full_html=True,
        )
        print(f"Saved chart: {chart_path}")
else:
    print("Chart HTML export is disabled. Set SAVE_CHART_HTML = True to enable it.")


Saved summary: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs\evaluation_summary.csv
Saved chart: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs\evaluation_charts\01_carbon_ratio.html
Saved chart: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs\evaluation_charts\02_carbon_comfort_tradeoff.html
Saved chart: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs\evaluation_charts\03_building_discomfort.html
Saved chart: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS5546 AI Planning and Decision Making\02Assignment\outputs\evaluation_charts\04_comfort_range.html
Saved chart: C:\Users\tanka\Dropbox\08-1 NUS Master in Computing AI specialization\CS

## Evaluation reminder

Changing `EPISODES` gives every learning controller the same episode budget, but it does not create a held-out evaluation. Each runner currently uses the final deterministic episode on the same simulation period used during training. For final research claims, repeat the workflow across several seeds and evaluate saved policies on a separate test period.
